# PyTorch 计算机视觉

参考 Notebook：  
https://github.com/mrdbourke/pytorch-deep-learning/blob/main/03_pytorch_computer_vision.ipynb

在线教材参考：  
https://www.learnpytorch.io/03_pytorch_computer_vision/

IBM 参考资料：  
https://www.ibm.com/think/topics/convolutional-neural-networks#763338460

## 本节内容概览

- 使用 `torchvision.datasets` 获取用于计算机视觉任务的数据集。
- PyTorch 中**卷积神经网络（Convolutional Neural Network，CNN）** 的基本架构。
- 一个端到端的多分类图像分类问题。
- 使用 **PyTorch CNN** 建模的基本步骤：
  - 使用 PyTorch 创建 CNN 模型。
  - 选择损失函数和优化器。
  - 训练模型。
  - 评估模型。

## 什么是卷积神经网络（Convolutional Neural Network，CNN）

![alt text](source/03-cnn.png)

## 0. PyTorch 中的计算机视觉相关库

- `torchvision.datasets`：提供常用计算机视觉数据集及其加载功能。
- `torchvision.models`：提供预训练的计算机视觉模型，可直接用于迁移学习或自己的任务。
- `torchvision.transforms`：提供图像预处理与数据增强功能，使图像适合输入机器学习模型。
- `torch.utils.data.Dataset`：PyTorch 数据集的基类，用于自定义数据集。
- `torch.utils.data.DataLoader`：用于将数据集封装为可迭代对象，支持批量加载（Batch）、打乱数据（Shuffle）等功能。

In [1]:
# Import PyTorch
import torch
import torch.nn as nn

# Import torchvision
import torchvision
from torchvision import datasets, transforms
from torchvision import transforms
from torchvision.transforms import ToTensor

# Import matplotlib for visualization
import matplotlib.pyplot as plt

# Check versions
print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

## 1. 获取数据集

本节将使用 `torchvision.datasets` 提供的**FashionMNIST**数据集。

数据集文档：

https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html

![FashionMNIST 数据集](source/03-Fashion-MNIST-dataset-cover.png)

In [2]:
# Setup training data
from torchvision import datasets
train_data = datasets.FashionMNIST(
    root = "data", # where to download data to?
    train = True, # do we want the training dataset?
    download = True, # do we want to download yes/no?
    transform = torchvision.transforms.ToTensor(), # how do we want to transform the data?
    target_transform = None # how do we want to transform the labels/targets?
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

### 将图像转换为 Tensor

`torchvision.transforms.ToTensor()` 用于将图像转换为 PyTorch 的 `Tensor`。

官方文档：

https://docs.pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html

In [3]:
len(train_data), len(test_data)

In [4]:
# See the first training example
image, label = train_data[0]
image, label

In [5]:
class_names = train_data.classes

class_names

In [6]:
classes_to_idx = train_data.class_to_idx
classes_to_idx

### 1.1 检查数据的输入与输出形状（Shape）

In [7]:
# Check the shape of our image
print(f"Image shape: [Color_channels, height, width] -> {image.shape} ")
print(f"Image label: {class_names[label]}")

### 1.2 数据可视化

In [8]:
import matplotlib.pyplot as plt
image, label = train_data[0]
print(f"Image shape: {image.shape}")
plt.imshow(image.squeeze())
plt.title(label)
# image 

In [9]:
plt.imshow(image.squeeze(), cmap="gray")
plt.title(class_names[label])
plt.axis(False)

> **思考：**
>
> 你认为这些服饰图像能够仅通过**线性模型（Linear Model）** 来建模吗？
>
> 还是说，我们需要引入**非线性（Non-linearity）** 才能更好地学习这些图像中的复杂特征？

In [10]:
train_data, test_data

## 2. 准备 DataLoader

目前，我们的数据还是 PyTorch `Dataset` 的形式。

`DataLoader` 可以将数据集转换为 Python 可迭代对象（Iterable）。

更具体地说，我们希望将数据划分为一个个批次（Batch）或小批次（Mini-batch）。

为什么要这样做？

1. **提高计算效率**  
   计算设备通常无法一次性查看或存储 60000 张图像，因此我们将数据拆分成更小的批次，例如每次处理 32 张图像（Batch Size = 32）。

2. **增加模型更新梯度的机会**  
   使用 Mini-batch 后，神经网络在每个 Epoch 中可以进行多次梯度更新，从而更频繁地调整模型参数。

In [11]:
from torch.utils.data import DataLoader

# Set up the batch size hyperparameter
BATCH_SIZE = 32

# Turn datasets into iterable DataLoaders
train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

test_dataloader, train_dataloader

print(f"DataLoader: {train_dataloader, test_dataloader}")
print(f"Length of training dataloader: {len(train_dataloader)} batches of {BATCH_SIZE} samples")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE} samples")

In [12]:
# Check out what inside the training dataloader
train_features_batch, train_labels_batch = next(iter(train_dataloader))
train_features_batch.shape, train_labels_batch.shape

In [13]:
# Show a sample 
torch.manual_seed(42)
random_idx = torch.randint(0, len(train_features_batch), size=[1]).item()
img, label = train_features_batch[random_idx], train_labels_batch[random_idx]
plt.imshow(img.squeeze(), cmap="gray")
plt.title(class_names[label])
plt.axis(False)

print(f"Image shape: {img.shape}")
print(f"Label: {label}, label size: {label.shape}, class name: {class_names[label]}")

## 3. Model 0：构建基线模型

当我们开始进行一系列机器学习建模实验时，最佳实践是先从一个**基线模型（Baseline Model）**开始。

基线模型是一个相对简单的模型，后续的模型和实验都会以它为基础进行改进。

换句话说：

> **先从简单模型开始，必要时再逐步增加复杂度。**

In [14]:
# Create a flatten layer
flatten_model = nn.Flatten()

# Get a sample image
x = train_features_batch[0]

# Flatten the sample
output = flatten_model(x) # performs a forward pass of the sample through the flatten layer

# Print out what happened
print(f"Original shape: {x.shape} -> [Color_channels, height, width]")
print(f"Flattened shape: {output.shape} -> [Color_channels * height * width]")

In [ ]:
from torch import nn
# Create a flatten layer
class FashionMNISTModelV0(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape, out_features=hidden_units),
            nn.Linear(in_features=hidden_units, out_features=output_shape)
        )
    
    def forward(self, x):
        return self.layer_stack(x)

In [16]:
torch.manual_seed(42)

# Setup model with input and output size
model_0 = FashionMNISTModelV0(input_shape=28 * 28, 
                              hidden_units=10, 
                              output_shape=len(class_names)).to("cpu")
model_0

dummy_x = torch.rand(1, 1, 28, 28) # create a random tensor with the same shape as a single image
model_0(dummy_x).shape # pass the random tensor through the model and check the output shape

In [17]:
model_0.state_dict()

### 3.1 设置损失函数、优化器和评估指标

- **损失函数（Loss Function）**  
  由于这是一个**多分类任务（Multi-class Classification）**，因此使用 `nn.CrossEntropyLoss()`。

- **优化器（Optimizer）**  
  使用 `torch.optim.SGD()`（随机梯度下降，Stochastic Gradient Descent）。

- **评估指标（Evaluation Metric）**  
  由于这是分类问题，我们使用**准确率（Accuracy）** 作为模型的评估指标。


In [18]:
import requests
import base64
from pathlib import Path 

# Download helper functions from Learn PyTorch repo(if it's not already downloaded)
if Path("ml_helper_functions.py").is_file():
    print("ml_helper_functions.py already exists, skipping download.")
else:
    print("Downloading ml_helper_functions.py...")
    url = "https://raw.githubusercontent.com/LiuAoran/lar_ml_helpers/refs/heads/main/ml_helper_functions.py"
    request = requests.get(url)
    with open("ml_helper_functions.py", "wb") as f:
        f.write(request.content)
        print("ml_helper_functions.py downloaded successfully.")

In [ ]:
# import accuracy metric helper function
from ml_helper_functions import accuracy_fn

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.1)

### 3.2 创建一个用于统计实验耗时的函数

机器学习是一个高度依赖实验的过程。

在实验中，我们通常需要关注两件事：

1. **模型性能（Model Performance）**  
   例如：损失值（Loss）、准确率（Accuracy）等指标。

2. **运行速度（Runtime）**  
   即模型训练和推理所花费的时间。

In [ ]:
from timeit import default_timer as timer
def print_train_time(start: float, end: float, device: torch.device = "cpu"): # type: ignore
    total_time = end - start
    print(f"Train time on {device}: {total_time:.3f} seconds")
    return total_time

In [21]:
start_time = timer()

# Some Code to train the model would go here...

end_time = timer()

print_train_time(start_time, end_time, device = "cpu") # type: ignore

> **重点：** 优化器（Optimizer）是在**每个 Batch** 完成反向传播后更新一次模型参数，而**不是每个 Epoch** 结束后才更新一次。

## 3.3 创建训练循环，并使用批量数据训练模型

1. 遍历所有 Epoch。

2. 遍历训练数据的每个 Batch，执行训练步骤，并计算**每个 Batch 的训练损失**。

3. 遍历测试数据的每个 Batch，执行测试步骤，并计算**每个 Batch 的测试损失**。

4. 打印训练过程中的关键信息。

5. 统计整个运行过程的耗时。

In [22]:
# impport tqdm for progress bar
from tqdm.auto import tqdm

# Set the seed for reproducibility
torch.manual_seed(42)
train_time_start_on_cpu = timer()

# Check the model's device
print(f"Model is on: {next(model_0.parameters()).device}")

# Set the number of epochs (how many times the model will look at the data)
epochs = 3

# Create a training and testing loop
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")

    # Training
    train_loss = 0

    # Add a loop to loop through the training batches
    for batch, (X, y) in enumerate(train_dataloader):
        model_0.train() 

        y_pred = model_0(X) # Do the forward pass

        loss = loss_fn(y_pred, y) # Calculate the loss
        train_loss += loss.item() # add the loss to the total training loss

        optimizer.zero_grad() # Optimizer zero grad

        loss.backward() # Loss backward

        optimizer.step() # Optimizer step

        #Print out what's happening
        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(train_dataloader.dataset)} samples.")  # type: ignore

    # Testing
    test_loss, test_acc = 0, 0
    model_0.eval() # Put model in evaluation mode
    with torch.inference_mode(): # Turn on inference mode (turns off gradients)
        for X_test, y_test in test_dataloader:

            # 1. Forward pass
            test_pred = model_0(X_test)

            # 2. Calculate loss(accumulatively)
            test_loss += loss_fn(test_pred, y_test).item()
        
            # 3. Calculate accuracy(accumulatively)
            test_acc += accuracy_fn(y_true=y_test, y_pred=test_pred.argmax(dim=1))

        # Calculate the average loss and accuracy per batch
        test_loss /= len(test_dataloader)
        test_acc /= len(test_dataloader)
    
    # Print out what's happening
    print(f"Train loss: {train_loss:.5f} | Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%\n")

train_time_end_on_cpu = timer()
train_time_on_cpu_model_0 = print_train_time(train_time_start_on_cpu, train_time_end_on_cpu, device=str(next(model_0.parameters()).device)) # type: ignore
            

## 4. 进行预测并获取 Model 0 的结果

In [23]:
torch.manual_seed(42)

def evaluate_model(model: torch.nn.Module, 
                   data_loader: torch.utils.data.DataLoader, 
                   loss_fn: torch.nn.Module, accuracy_fn):
    """Returns a dictionary containing the results of model predicting on data_loader."""
    loss, acc = 0, 0
    model.eval() # Put model in evaluation mode
    with torch.inference_mode(): # Turn on inference mode (turns off gradients)
        for X, y in data_loader:
            # Make sure the model is on the right device
            X, y = X.to(next(model.parameters()).device), y.to(next(model.parameters()).device)
            # 1. Forward pass
            y_pred = model(X)

            # 2. Calculate loss(accumulatively)
            loss += loss_fn(y_pred, y)
            acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))
        
        # Scale loss and accuracy per batch
        loss /= len(data_loader)
        acc /= len(data_loader)
    
    return {"model_name": model.__class__.__name__, 
            "model_loss": loss.item(),  
            "model_acc": acc}

# Calculate the results of model_0 on the test dataset
model_0_results = evaluate_model(model=model_0,
                               data_loader=test_dataloader,
                               loss_fn=loss_fn,
                               accuracy_fn=accuracy_fn)

model_0_results

## 5. 设置与设备无关（Device-Agnostic）的代码

使程序在有 GPU 可用时自动使用 GPU，否则默认使用 CPU。

In [24]:
!nvidia-smi

In [25]:
import torch
import torch.nn as nn
torch.cuda.is_available()
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 6. Model 1：构建一个带有非线性的更好模型

我们已经在 Notebook 02 中学习过非线性（Non-linearity）的作用：

https://github.com/LiuAoran/freecodecamp-machine-learning-notes/blob/main/02_pytorch_classification.ipynb

In [26]:
# Create a model with non-linear and linear layers
class FashionMNISTModelV1(nn.Module):
    def __init__(self, 
                 input_shape: int, 
                 hidden_units: int, 
                 output_shape: int):
         super().__init__()
         self.layer_stack = nn.Sequential(
             nn.Flatten(), # Flatten the input image into a single vector
             nn.Linear(in_features=input_shape, out_features=hidden_units),
             nn.ReLU(),
             nn.Linear(in_features=hidden_units, out_features=output_shape),
             nn.ReLU()
         )
         
    def forward(self, x):
        return self.layer_stack(x)

In [27]:
# Create a model instance and send it to the target device
torch.manual_seed(42)
model_1 = FashionMNISTModelV1(input_shape=28*28,
                              hidden_units=10,
                              output_shape=len(class_names)).to(device)
next(model_1.parameters()).device

### 6.1 设置损失函数、优化器和评估指标

In [28]:
from ml_helper_functions import accuracy_fn
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_1.parameters(), lr=0.1)

## 6.2 将训练循环和评估/测试循环封装为函数

接下来，我们创建两个函数：

- 训练循环函数：`train_step()`
- 测试循环函数：`test_step()`

In [29]:
def train_step(
    model: torch.nn.Module, 
    data_loader: torch.utils.data.DataLoader, 
    loss_fn: torch.nn.Module, 
    optimizer: torch.optim.Optimizer, 
    accuracy_fn,
    device: torch.device = device # type: ignore
):
    """Performs a training step with the 
    given model, data loader, loss function, optimizer, and accuracy function."""
    ### Training Step ###
    train_loss, train_acc = 0, 0
    # Put the model in training mode
    model.train()
    
    for batch, (X, y) in enumerate(data_loader):

        X, y = X.to(device), y.to(device)

        y_pred = model(X) # Do the forward pass (Outputs the raw logits from the model)

        loss = loss_fn(y_pred, y) # Calculate the loss
        train_loss += loss.item() # add the loss to the total training loss
        train_acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1)) # go from logits -> predicted labels

        optimizer.zero_grad() # Optimizer zero grad

        loss.backward() # Loss backward

        optimizer.step() # Optimizer step

        #Print out what's happening
        if batch % 400 == 0:
            print(f"Looked at {batch * len(X)}/{len(data_loader.dataset)} samples.")  # type: ignore

    # Devide the total training loss and accuracy by the number of batches
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)
    print(f"Train loss: {train_loss:.5f} | Train accuracy: {train_acc:.2f}%")
    

In [ ]:
def test_step(
    model: torch.nn.Module, 
    data_loader: torch.utils.data.DataLoader, 
    loss_fn: torch.nn.Module, 
    accuracy_fn,
    device: torch.device = device # type: ignore
):
    """Performs a testing step with the given model, data loader, loss function, and accuracy function."""
    ### Testing Step ###
    test_loss, test_acc = 0, 0
    # Put the model in evaluation mode
    model.eval()
    
    with torch.inference_mode(): # Turn on inference mode (turns off gradients)
        for X, y in data_loader:

            X, y = X.to(device), y.to(device)

            test_pred = model(X) # Do the forward pass

            loss = loss_fn(test_pred, y) # Calculate the loss
            test_loss += loss.item() # add the loss to the total test loss
            test_acc += accuracy_fn(y_true=y, y_pred=test_pred.argmax(dim=1)) # go from logits -> predicted labels

        # Devide the total test loss and accuracy by the number of batches
        test_loss /= len(data_loader)
        test_acc /= len(data_loader)
        print(f"Test loss: {test_loss:.5f} | Test accuracy: {test_acc:.2f}%")

In [31]:
torch.manual_seed(42)

# Measure the time it takes to train and test the model on the CPU
from timeit import default_timer as timer
train_time_start_on_gpu = timer()

# Set epochs
epochs = 3

# Create a optimization and evaluation loop using the train_step() and test_step() functions
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_step(model=model_1,
               data_loader=train_dataloader,
               loss_fn=loss_fn,
               optimizer=optimizer,
               accuracy_fn=accuracy_fn,
               device=device) # type: ignore
    
    test_step(model=model_1,
              data_loader=test_dataloader,
              loss_fn=loss_fn,
              accuracy_fn=accuracy_fn,
              device=device) # type: ignore
    
train_time_end_on_gpu = timer()
train_time_on_gpu_model_1 = print_train_time(train_time_start_on_gpu, train_time_end_on_gpu, device=device) # type: ignore
    

In [32]:
model_0_results

> **注意：**
>
> 有时，根据数据规模和硬件配置的不同，你可能会发现模型在 **CPU** 上训练反而比 **GPU** 更快。
>
> 可能的原因包括：
>
> 1. **数据和模型在 CPU 与 GPU 之间传输存在额外开销**，当数据量较小时，这部分开销可能超过 GPU 带来的计算加速。
>
> 2. **当前设备的 CPU 计算能力强于 GPU**，因此在某些任务上 CPU 的运行速度反而更快。
>
> 相关阅读：
>
> **Making Deep Learning Go Brrrr From First Principles**
>
> https://horace.io/brrr_intro.html

In [33]:
# Train time on CPU for model_0
train_time_on_cpu_model_0

In [34]:
model_1_results = evaluate_model(model=model_1,
                               data_loader=test_dataloader, 
                               loss_fn=loss_fn,
                               accuracy_fn=accuracy_fn)
model_1_results

## 7. Model 2：构建卷积神经网络（CNN）

CNN 也被称为 ConvNet。

CNN 以擅长从视觉数据中发现模式而闻名，尤其适用于图像相关任务。

如果想了解 CNN 内部发生了什么，可以参考这个可视化网站：

https://poloclub.github.io/cnn-explainer/

In [ ]:
# Create a model with non-linear and linear layers
class FashionMNISTModelV2(nn.Module):
    """
    Model architecture that replicates tiny VGG (Visual Geometry Group) model.
    model from CNN explainer website: https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, 
                 input_shape: int, 
                 hidden_units: int, 
                 output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
             nn.Conv2d(in_channels=1, 
                       out_channels=hidden_units, 
                       kernel_size=3, 
                       stride=1, 
                       padding=1), 
             # Values we can set ourselfs in our NN's are called hyperparameters, the values we can't set ourselves are called parameters
             nn.ReLU(),
             nn.Conv2d(in_channels=hidden_units, 
                       out_channels=hidden_units, 
                       kernel_size=3, 
                       stride=1, 
                       padding=1),
             nn.ReLU(),
             nn.MaxPool2d(kernel_size=2, stride=2)
        )
         
        self.conv_block_2 = nn.Sequential(
             nn.Conv2d(in_channels=hidden_units, 
                       out_channels=hidden_units, 
                       kernel_size=3, 
                       stride=1, 
                       padding=1),
             nn.ReLU(),
             nn.Conv2d(in_channels=hidden_units, 
                       out_channels=hidden_units, 
                       kernel_size=3, 
                       stride=1, 
                       padding=1),
             nn.ReLU(),
             nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
             nn.Flatten(),
             nn.Linear(in_features=hidden_units * 7 * 7, # The input features to the linear layer is the number of channels * the height * the width of the output from the previous conv block 
                       out_features=output_shape)
        )

    def forward(self, x):
         x = self.conv_block_1(x)
         # print(f"Shape after conv_block_1: {x.shape}")
         x = self.conv_block_2(x)
         # print(f"Shape after conv_block_2: {x.shape}")
         x = self.classifier(x)
         return x


In [36]:
torch.manual_seed(42)
# Create a model with non-linear and linear layers
model_2 = FashionMNISTModelV2(input_shape=1,
                              hidden_units=10,
                              output_shape=len(class_names)).to(device)

In [37]:
rand_image_tensor = torch.randn(size=(1, 28, 28)).to(device)
rand_image_tensor.shape

In [38]:
model_2(rand_image_tensor.unsqueeze(0)).to(device)

### 7.1 深入理解 `nn.Conv2d()`

参考官方文档：

https://docs.pytorch.org/docs/2.12/generated/torch.nn.Conv2d.html

In [39]:
torch.manual_seed(42)

images = torch.randn(size = (32, 3, 64, 64))
test_image = images[0]

print(f"Image batch shape: {images.shape}")
print(f"Single image shape: {test_image.shape}")
print(f"Test image shape: {test_image.shape}")
print(f"Test image: \n {test_image}")

In [40]:
# Create a single convolutional layer
conv_layer = nn.Conv2d(in_channels=3, # input channels
                       out_channels=10, # output channels
                       kernel_size= 3, # size of the kernel/filter
                       stride= 1, # how many pixels the kernel moves at a time
                       padding= 0) # how many pixels to pad the input image with

conv_output = conv_layer(test_image.unsqueeze(0)) # unsqueeze adds a batch dimension to the image
conv_output.shape
conv_output

In [41]:
test_image.unsqueeze(0).shape

In [42]:
torch.__version__

### 7.2 深入理解 `nn.MaxPool2d()`

参考官方文档：

https://docs.pytorch.org/docs/2.12/generated/torch.nn.MaxPool2d.html

In [43]:
test_image.shape

In [44]:
# Print out original image shape without unsqueezed dimension

print(f"Original image shape: {test_image.shape} -> [Color_channels, height, width]")
print(f"Test image with batch dimension shape: {test_image.unsqueeze(0).shape} -> [Batch_size, Color_channels, height, width]")

# Create a sample MaxPool2d layer
max_pool_layer = nn.MaxPool2d(kernel_size=2)

# Pass data through just the conv_layer
test_image_conv = conv_layer(test_image.unsqueeze(dim=0))
print(f"Shape after conv_layer: {test_image_conv.shape} -> [Batch_size, Color_channels, height, width]")

# Pass data through the max_pool_layer
test_image_max_pool = max_pool_layer(test_image_conv)
print(f"Shape after max_pool_layer: {test_image_max_pool.shape} -> [Batch_size, Color_channels, height, width]")

In [45]:
torch.manual_seed(42)

random_tensor = torch.randn(size=(1, 1, 2, 2))

# Create a sample MaxPool2d layer
max_pool_layer = nn.MaxPool2d(kernel_size=2)

# Pass data through the max_pool_layer
max_pool_tensor = max_pool_layer(random_tensor)

print(f"max_pool_layer input: \n {random_tensor} \n")
print(f"max_pool_layer input shape: {random_tensor.shape} -> [Batch_size, Color_channels, height, width]")
print(f"max_pool_layer output: \n {max_pool_tensor} \n")
print(f"max_pool_layer output shape: {max_pool_tensor.shape} -> [Batch_size, Color_channels, height, width]")


### 7.3 为 Model 2 设置损失函数和优化器

In [46]:
# Setup loss function and optimizer for model_2
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.1)

使用训练函数和测试函数训练并测试 `model_2`

In [47]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Measure the time it takes to train and test the model on the GPU
from timeit import default_timer as timer
train_time_start_2 = timer()

# train the model on the GPU
epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_step(model=model_2,
               data_loader=train_dataloader,
               loss_fn=loss_fn,
               optimizer=optimizer,
               accuracy_fn=accuracy_fn,
               device=device) # type: ignore
    
    test_step(model=model_2,
              data_loader=test_dataloader,
              loss_fn=loss_fn,
              accuracy_fn=accuracy_fn,
              device=device) # type: ignore
    
    train_time_end_2 = timer()
train_time_on_gpu_model_2 = print_train_time(train_time_start_2, train_time_end_2, device=device) # type: ignore



In [ ]:
model_2_results = evaluate_model(model=model_2,
                                 data_loader=test_dataloader,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn)

model_2_results

## 8. 比较各模型的性能与训练耗时

In [ ]:
import pandas as pd

comparison_dict = {
    "model_0": model_0_results,
    "model_1": model_1_results,
    "model_2": model_2_results
}

compare_df = pd.DataFrame(comparison_dict).T

compare_df

In [ ]:
# Compare the training times of the three models
compare_df["train_time"] = {
    "model_0": train_time_on_cpu_model_0,
    "model_1": train_time_on_gpu_model_1,
    "model_2": train_time_on_gpu_model_2
}

compare_df

In [ ]:
# Visualize the results of the three models
compare_df[["model_acc"]].plot(kind="barh", figsize=(10, 5), title="Model Accuracy Comparison")
plt.xlabel("Accuracy(%)")
plt.ylabel("Models")
plt.show()


## 9. 使用最佳模型进行随机预测并评估结果

In [ ]:
def make_predictions(model: torch.nn.Module, data: list, device: torch.device = device): # type: ignore
    """Makes predictions on a list of images using a trained PyTorch model."""
    model.to(device) # Send the model to the target device
    model.eval() # Put the model in evaluation mode
    pred_probs = []
    with torch.inference_mode(): # Turn on inference mode (turns off gradients)
        for sample in data:
            # Prepare the sample
            sample = sample.to(device).unsqueeze(0) # Add a batch dimension and send to device
            
            # Make a prediction
            pred_logit = model(sample) # Get raw logits from the model
            
            # Convert logits to prediction probabilities
            pred_prob = torch.softmax(pred_logit, dim=1) # Apply softmax to get probabilities
            
            pred_probs.append(pred_prob.cpu()) # Move to CPU and append to list
    
    return torch.stack(pred_probs) # Concatenate all prediction probabilities into a single tensor

In [ ]:
import random
# random.seed(42)

test_samples = []
test_labels = []
for sample, label in random.sample(list(test_data), 9): # type: ignore
    test_samples.append(sample)
    test_labels.append(label)

# View the shape of the test samples and labels
print(f"Test samples shape: {test_samples[0].shape}")


In [ ]:
plt.imshow(test_samples[0].squeeze(), cmap="gray")
plt.title(f"{class_names[test_labels[0]]}")

In [ ]:
# Make predictions on the test samples using model_2
pred_probs = make_predictions(model=model_2, data=test_samples, device=device) # type: ignore
pred_probs[:2]

In [ ]:
# Convert prediction probabilities to predicted labels
pred_labels = pred_probs.argmax(dim=-1).squeeze()
print(f"Predicted labels: {pred_labels}")
print(f"True labels: {torch.tensor(test_labels)}")

In [ ]:
# Plot the test samples and their predicted labels
plt.figure(figsize=(9, 9))
nrows, ncols = 3, 3
for i, sample in enumerate(test_samples):
    plt.subplot(3, 3, i + 1) # Create a subplot for each sample
    plt.imshow(sample.squeeze(), cmap="gray") # Show the sample image
    pred_label = class_names[pred_labels[i]]
    true_label = class_names[test_labels[i]]
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}") # Add a title with the predicted and true labels
    if pred_label == true_label:
        plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color="green") # If the prediction is correct, make the title green
    else:
        plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color="red") # If the prediction is incorrect, make the title red  
    plt.axis(False) # Turn off the axis
plt.tight_layout()


## 10. 绘制混淆矩阵，进一步评估模型预测结果

混淆矩阵（Confusion Matrix） 是分类模型常用的评价方法，通过统计模型预测结果与真实标签之间的对应关系来评估模型性能。

在二分类任务中，混淆矩阵包含四种情况：TP（真正例）、TN（真负例）、FP（假正例） 和 FN（假负例）。

基于混淆矩阵可以计算多种评价指标，如准确率（Accuracy）、精确率（Precision）、召回率（Recall） 和 F1-score。

相比仅使用准确率，混淆矩阵能够更全面地反映模型的分类效果，尤其适用于分析类别不平衡问题以及模型容易混淆的类别，是评价分类模型性能的重要工具。

1. 使用训练好的模型在测试数据集上进行预测。

2. 使用 `torchmetrics.ConfusionMatrix` 创建混淆矩阵。

3. 使用 `mlxtend.plotting.plot_confusion_matrix()` 绘制混淆矩阵。

In [ ]:
# import tqdm.auto
from tqdm.auto import tqdm

# 1. Make predictions on the test samples using model_2
y_preds = []
model_2.eval() # Put the model in evaluation mode
with torch.inference_mode(): # Turn on inference mode (turns off gradients)
    for X, y in tqdm(test_dataloader):
        X, y = X.to(device), y.to(device)
        y_logits = model_2(X) # Get raw logits from the model
        y_pred = torch.softmax(y_logits.squeeze(), dim=0).argmax(dim=1) # Apply softmax to get prediction probabilities
        y_preds.append(y_pred.cpu())

# Concatenate all predictions into a single tensor
print(f"Number of batches predicted: {len(y_preds)}")  
y_preds_tensor = torch.cat(y_preds)
print(f"Shape of predictions tensor: {y_preds_tensor.shape}")

In [ ]:
# See if requered packages are installed and import them if not
try:
    import torchmetrics, mlxtend
    print("torchmetrics and mlxtend are already installed.")
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchmetrics", "mlxtend"])
    import torchmetrics, mlxtend
    print("torchmetrics and mlxtend have been installed.")

# Check the versions of torchmetrics and mlxtend
print(f"torchmetrics version: {torchmetrics.__version__}")
print(f"mlxtend version: {mlxtend.__version__}")

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

# Set up the confusion matrix
confmat = ConfusionMatrix(num_classes=len(class_names), task="multiclass").to(device)
confmat_tensor = confmat(
    preds=y_preds_tensor.to(device),
    target=test_data.targets.to(device))
confmat_tensor

# Plot the confusion matrix
fig, ax = plot_confusion_matrix(conf_mat=confmat_tensor.cpu().numpy(),
                                class_names=class_names,
                                figsize=(10, 7))


## 11. 保存并加载性能最佳的模型

In [ ]:
from pathlib import Path
# Create a directory to save the model if it doesn't exist
model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_name = "fashion_mnist_model_2.pth"
model_path = model_dir / model_name

print(f"Model will be saved to: {model_path}")
torch.save(obj=model_2.state_dict(), f=model_path)

model_2.state_dict()


In [ ]:
# Create a new instance of the model and load the saved state_dict
loaded_model_2 = FashionMNISTModelV2(input_shape=1,
                                        hidden_units=10,
                                        output_shape=len(class_names)).to(device)
loaded_model_2.load_state_dict(torch.load(f=model_path))

loaded_model_2.to(device) # Send the model to the target device

In [ ]:
# Evaluate the loaded model on the test dataset
loaded_model_2_results = evaluate_model(model=loaded_model_2,
                                        data_loader=test_dataloader,
                                        loss_fn=loss_fn,
                                        accuracy_fn=accuracy_fn)

torch.isclose(torch.tensor(model_2_results["model_loss"]), torch.tensor(loaded_model_2_results["model_loss"])) and \
torch.isclose(torch.tensor(model_2_results["model_acc"]), torch.tensor(loaded_model_2_results["model_acc"]))